# MetaTool Tool-Selection Agent — Colab Runner

End-to-end notebook for the MetaTool benchmark assignment. Runs three open-source LLMs (Gemma 2 2B-it, Llama 3.2 3B-Instruct, Qwen 2.5 7B-Instruct in 4-bit) sequentially on a balanced sample, then computes per-model metrics.

**Hardware:** designed for a free-tier Colab T4 (~15 GB VRAM). Models are loaded and unloaded one at a time.

**Before running:**
1. Runtime → Change runtime type → GPU (T4).
2. You will need a Hugging Face token with access to `meta-llama/Llama-3.2-3B-Instruct` and `google/gemma-2-2b-it` (both are gated; accept the licenses on their HF model pages once).

## 1. Clone project + install dependencies

> **If you previously ran an older version of this notebook that pinned `torch==2.4.0`:** Colab's torch was downgraded and `torchvision`/`torchaudio` are now broken. Reset the runtime first via **Runtime → Disconnect and delete runtime**, then re-run from cell 1.

In [ ]:
# Clone the project repo (this notebook lives inside it).
import os, subprocess, sys
REPO_URL = 'https://github.com/jamelski1/hold-the-line.git'
BRANCH = 'claude/metatool-agent-assignment-ImZDo'
WORK_ROOT = '/content/hold-the-line'
PROJECT_DIR = f'{WORK_ROOT}/metatool-agent'

if not os.path.isdir(WORK_ROOT):
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, WORK_ROOT], check=True)
else:
    subprocess.run(['git', '-C', WORK_ROOT, 'pull'], check=True)
os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())

In [ ]:
# Install only what Colab is missing or needs a recent version of.
# IMPORTANT: do NOT pin torch / numpy / pandas. Colab ships modern versions
# and downgrading them silently breaks torchvision/torchaudio. We only need
# the four LLM/eval-side packages below.
!pip install -q -U transformers accelerate bitsandbytes rapidfuzz sentencepiece

In [ ]:
# Hugging Face login (needed for Gemma 2 + Llama 3.2 gated models).
from huggingface_hub import login
from getpass import getpass
import os
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('HF token: ')
login(os.environ['HF_TOKEN'])

## 2. Download MetaTool data

Clones the MetaTool repo (`HowieHwong/MetaTool`) and copies `big_tool_des.json` and `all_clean_data.csv` into our `data/` directory. The cell prints the schema of each file before any sampling code runs, per the brief.

In [ ]:
import os, shutil, subprocess, json, glob
import pandas as pd

DATA_JSON = 'data/big_tool_des.json'
DATA_CSV  = 'data/all_clean_data.csv'

# If the files are already present in the cloned repo, skip the download.
if os.path.isfile(DATA_JSON) and os.path.isfile(DATA_CSV):
    print(f'Found existing data files — skipping MetaTool download.')
else:
    META_REPO = '/content/MetaTool'
    if not os.path.isdir(META_REPO):
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/HowieHwong/MetaTool.git', META_REPO], check=True)
    candidates_json = glob.glob(f'{META_REPO}/**/big_tool_des.json', recursive=True)
    candidates_csv  = glob.glob(f'{META_REPO}/**/all_clean_data.csv', recursive=True)
    assert candidates_json, 'big_tool_des.json not found in MetaTool repo'
    assert candidates_csv,  'all_clean_data.csv not found in MetaTool repo'
    src_json, src_csv = candidates_json[0], candidates_csv[0]
    print('Found:', src_json); print('Found:', src_csv)
    os.makedirs('data', exist_ok=True)
    shutil.copy(src_json, DATA_JSON)
    shutil.copy(src_csv,  DATA_CSV)
    print('Copied into data/')

In [ ]:
# Inspect schema BEFORE running sampling (per brief).
with open('data/big_tool_des.json') as f:
    raw = json.load(f)
print('big_tool_des.json root type:', type(raw).__name__, '| size:', len(raw))
if isinstance(raw, list):
    print('first entry keys:', list(raw[0].keys()) if isinstance(raw[0], dict) else type(raw[0]))
    print(json.dumps(raw[:2], indent=2)[:600])
elif isinstance(raw, dict):
    keys = list(raw.keys())
    print('first 5 tool names:', keys[:5])
    print('sample value:', json.dumps(raw[keys[0]], indent=2)[:300])

df = pd.read_csv('data/all_clean_data.csv')
print('\nall_clean_data.csv columns:', list(df.columns))
print('shape:', df.shape)
df.head()

## 3. Build sampled dataset

5 positives per tool (~240) + balanced curated negatives. Seeded with `RANDOM_SEED=42`.

In [ ]:
from pathlib import Path
from src.sampling import build_sampled_dataset

sampled = build_sampled_dataset(
    big_tool_des_path=Path('data/big_tool_des.json'),
    all_clean_data_path=Path('data/all_clean_data.csv'),
    out_path=Path('data/sampled_data.csv'),
    per_tool=5,
    n_negatives=None,  # balanced
    seed=42,
)
sampled.head()

In [ ]:
# Sanity check: per-tool counts and balance.
print('total:', len(sampled))
print('positive:', int(sampled['requires_tool'].sum()))
print('negative:', int((~sampled['requires_tool']).sum()))
print('\nper-tool counts (positives only):')
print(sampled[sampled['requires_tool']]['gold_tool'].value_counts().describe())

## 4. Render the prompt for one query

Visual sanity check: the same prompt is sent to all three models (only the chat template wrapper differs).

In [ ]:
from src.prompts import render_prompt
from src.sampling import load_tool_list
tools = load_tool_list(Path('data/big_tool_des.json'))
print(render_prompt(sampled.iloc[0]['query'], tools)[:2000])

## 5. Run all three models sequentially

Each model is loaded, run over the full sample, and unloaded before the next is loaded. Predictions and malformed-response logs are written to `results/`.

In [ ]:
from pathlib import Path
import pandas as pd
from src.agent import run_agent
from src.evaluate import evaluate_predictions, write_summary
from src.llm_clients import ALL_SPECS, build_client
from src.sampling import load_tool_list

tools = load_tool_list(Path('data/big_tool_des.json'))
sampled = pd.read_csv('data/sampled_data.csv')
results_dir = Path('results'); results_dir.mkdir(exist_ok=True)

metrics_all = []
for spec in ALL_SPECS:
    print(f'\n=== {spec.label} ===')
    client = build_client(spec)
    preds_path     = results_dir / f'predictions_{spec.label}.csv'
    malformed_path = results_dir / f'malformed_{spec.label}.jsonl'
    metrics_path   = results_dir / f'metrics_{spec.label}.json'

    run_agent(client, sampled, tools,
              out_predictions_path=preds_path,
              out_malformed_path=malformed_path,
              max_tokens=256)
    metrics_all.append(evaluate_predictions(preds_path, metrics_path, spec.label))

    client.unload(); del client

summary = write_summary(metrics_all, results_dir / 'summary.csv')
summary

## 6. Inspect a few representative failures per model

Use this to populate the error-analysis section of `report.md`.

In [ ]:
import pandas as pd
for spec in ALL_SPECS:
    p = f'results/predictions_{spec.label}.csv'
    df = pd.read_csv(p)
    print(f'\n=== {spec.label} — failures ===')
    df['pred_tool_needed_filled'] = df['pred_tool_needed'].fillna(False).astype(bool)
    wrong_binary = df[df['requires_tool'] != df['pred_tool_needed_filled']]
    wrong_tool = df[df['requires_tool'] & df['pred_tool_needed_filled'] &
                    (df['gold_tool'].astype(str).str.strip() != df['pred_tool_name'].fillna('').astype(str).str.strip())]
    malformed = df[df['parse_status'] == 'malformed']
    print(f'wrong-binary: {len(wrong_binary)} | wrong-tool: {len(wrong_tool)} | malformed: {len(malformed)}')
    if len(wrong_tool):
        print(wrong_tool[['query','gold_tool','pred_tool_name']].head(3).to_string())